[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C55_TSR_Autonomous_Driving_Course/01_datasets/01_datasets_taxonomy.ipynb)

# 01 · 数据集与标志分类体系（长尾 / Zipf / 层次标签 / 层次评测 / 标注一致性）

目标：把「类别体系」从一件拍脑袋的事，变成**可以计算、可以验证**的工程决策。

本 notebook 你会亲手实现：
1. **合成一个真实形态的 TSR 类别分布**（120 细类 · Zipf · 3 万实例，对标 TT100K 量级）
2. **Zipf 指数拟合**与 head/common/rare 划分，并算出「靠随机采集补长尾要采多少数据」
3. **类别爆炸的账**：扁平笛卡尔积 vs 层次分解，压缩比与**增量成本**
4. **层次标签的编码**（路径 / 祖先集合）
5. **层次感知评测** hP/hR/hF —— 证明「诚实的降级」比「自信的错答」得分更高
6. **标注一致性**：框匹配 + 分尺寸桶的 IoU agreement + Cohen's κ
7. ✏️ repeat factor sampling / 尾类合并 / **标注规范判定器** / 层次代价矩阵

> 心智模型：**把「类别」压到最少，把「属性」做到最全，把「几何证据」原样输出。**
> 长尾里有一部分是真实世界给的，另一部分是你自己的类别设计造出来的——先分清这两者。

## 1 · 合成一个真实形态的 TSR 类别分布

用 **Zipf 分布**合成：第 r 常见的类别，其实例数正比于 $r^{-s}$。
参数按 **TT100K** 的真实形态选：≈120 个细类出现过、≈30,000 个标志实例、$s\approx1.6$。

（TT100K 实际是「出现 221 类，但标准协议只评实例数 >100 的 45 类」——
下面你会看到为什么必须这么做。）

In [ ]:
import numpy as np, math, json, itertools, collections
rng = np.random.default_rng(5768)          # 5768 = GB 5768

N_CLS, S_ZIPF, TOTAL = 120, 1.6, 30000
ranks = np.arange(1, N_CLS + 1)
p_true = ranks.astype(float) ** (-S_ZIPF)
p_true = p_true / p_true.sum()
counts = rng.multinomial(TOTAL, p_true)     # 每个细类实际采到多少实例

# LVIS 式的三档划分（这里按实例数；真实 LVIS 按「出现该类的图像数」）
frequent = counts > 1000
common   = (counts >= 100) & (counts <= 1000)
rare     = counts < 100

print(f'总实例数 {counts.sum():,} · 细类数 {N_CLS}')
print(f"{'档位':<24s} {'类别数':>7s} {'实例数':>9s} {'实例占比':>9s}")
for name, mask in [('frequent (>1000)', frequent),
                   ('common   (100-1000)', common),
                   ('rare     (<100)', rare)]:
    print(f'{name:<24s} {mask.sum():>7d} {counts[mask].sum():>9,d} '
          f'{100*counts[mask].sum()/TOTAL:>8.1f}%')

print(f'\n最常见类 {counts.max():,} 个实例 · 最罕见类 {counts.min()} 个实例 · '
      f'头尾比 {counts.max()/max(counts.min(),1):.0f}x')
print(f'前 10 类占全部实例的 {100*counts[:10].sum()/TOTAL:.1f}%')
print(f'后 60 类合计只占     {100*counts[60:].sum()/TOTAL:.1f}%')

assert counts.sum() == TOTAL
assert counts[:10].sum() / TOTAL > 0.75
assert counts[60:].sum() / TOTAL < 0.05
assert rare.sum() > frequent.sum() * 5, '罕见类的**数量**远多于常见类——这是长尾的定义'
print(f'\n⚠️  {rare.sum()} 个 rare 类加起来还不到全部实例的 '
      f'{100*counts[rare].sum()/TOTAL:.0f}% —— 这就是为什么 TT100K 的标准协议')
print('    只评「实例数 >100」的 45 类：**其余的类连算 AP 的统计意义都没有**。')
print('    但它们在道路上真实存在，漏检一样会出事。这就是 TSR 长尾的残酷之处。')

## 2 · Zipf 指数拟合：算出「靠随机采集补长尾」要采多少数据

$$f(r)=f_1\cdot r^{-s}\quad\Longleftrightarrow\quad \log f(r)=\log f_1-s\log r$$

在 log-log 上是一条直线，斜率就是 $-s$。**拟合出 $s$ 之后可以做一件很有用的事：
外推「要让最尾部的类攒够 N 个样本，总数据量得是多少」。**

In [ ]:
nz = counts > 0
slope, intercept = np.polyfit(np.log(ranks[nz]), np.log(counts[nz]), 1)
s_hat = -slope
print(f'拟合的 Zipf 指数 s_hat = {s_hat:.3f}   (真值 {S_ZIPF})')
assert 1.4 <= s_hat <= 1.85, s_hat

print(f"\n{'rank':>6s} {'实测':>8s} {'Zipf 拟合':>10s}")
for r_ in [1, 3, 10, 30, 60, 100, 120]:
    fit = math.exp(intercept + slope * math.log(r_))
    print(f'{r_:>6d} {counts[r_-1]:>8d} {fit:>10.1f}')

# —— 关键外推：靠**随机采集**把最尾部的类补到 100 个实例，需要多大的数据集？
TARGET_PER_CLASS = 100
need_total = TARGET_PER_CLASS / p_true[-1]
print(f'\n最罕见类的出现概率 p = {p_true[-1]:.2e}')
print(f'要让它攒够 {TARGET_PER_CLASS} 个实例，需要随机采集 **{need_total:,.0f}** 个标志实例')
print(f'  = 当前数据集的 **{need_total/TOTAL:.1f} 倍**')
assert need_total > 10 * TOTAL

print(f'\n而定向挖掘（C58：嵌入检索 + 场景标签 + 主动学习触发）'
      f'只需要 ~{TARGET_PER_CLASS} 个命中样本。')
print('✅ 结论：**长尾不能靠「多采一点」解决，必须靠定向挖掘 + 类别体系设计。**')
print('   幂律的性质就是：要把尾部提升一个数量级，总量得提升同一个数量级。')

## 3 · 类别爆炸的账：扁平笛卡尔积 vs 层次分解

把「限速」这一支的语义维度摊开，看看扁平体系的类别数是怎么炸的，
以及层次分解把它压到了多少。**最重要的是「增量成本」那一段。**

In [ ]:
SPEC = {
    'coarse':          7,    # 禁令/警告/指示/指路/辅助/施工/可变电子牌
    'fine':           60,    # 细类总数（其中 1 个是「限速族」的占位）
    'speed_values':   14,    # 5,10,15,20,30,...,120 km/h
    'speed_variants':  5,    # 限速 / 解除 / 最低限速 / 区间起 / 区间终
    'carrier':         2,    # 实体反光牌 / LED 可变电子牌
    'aux_slots':       4,    # 车型? 时段? 距离? 车道?  各有/无
}

def flat_class_count(spec):
    '''扁平体系：所有语义维度做笛卡尔积，每个组合是一个独立类别。'''
    speed_family = spec['speed_values'] * spec['speed_variants']
    non_speed    = spec['fine'] - 1
    return (non_speed + speed_family) * spec['carrier'] * (2 ** spec['aux_slots'])

def hier_output_dim(spec):
    '''层次体系：需要学的输出维度 = 各个头的维度之**和**（而不是积）。'''
    return (spec['coarse'] + spec['fine'] + spec['speed_values']
            + spec['speed_variants'] + spec['carrier'] + spec['aux_slots'])

flat, hier = flat_class_count(SPEC), hier_output_dim(SPEC)
print(f'扁平类别数        {flat:>8,d}')
print(f'层次输出维度      {hier:>8,d}')
print(f'压缩比            {flat/hier:>8.1f}x')
assert flat == 4128 and hier == 92
assert flat / hier > 20

# —— 增量成本：新增一个限速值（比如「限速 25」）
spec2 = dict(SPEC); spec2['speed_values'] += 1
d_flat = flat_class_count(spec2) - flat
d_hier = hier_output_dim(spec2) - hier
print('\n新增一个限速值（如「限速 25」）：')
print(f'  扁平体系 → 新增 **{d_flat} 个类别**，每个都要独立凑样本、独立算 AP、独立做 badcase')
print(f'  层次体系 → 属性头多 **{d_hier} 个分箱**，检测器与粗/细类头**完全不动**')
assert d_flat == 160 and d_hier == 1

# —— 人造长尾 vs 真实长尾
print(f'\n把 {TOTAL:,} 个实例摊到 {flat:,} 个扁平类别上：')
print(f'  平均每类 {TOTAL/flat:.1f} 个实例 —— **绝大多数类别一辈子凑不齐 10 个样本**')
print(f'摊到 {hier} 维层次输出上：平均每维 {TOTAL/hier:.0f} 个实例')
assert TOTAL / flat < 10 and TOTAL / hier > 300
print('\n✅ **人造长尾**（类别设计造出来的）可以设计掉；')
print('   **真实长尾**（「注意牲畜」本来就罕见）只能靠数据闭环（C58）。先分清这两者。')

## 4 · 层次标签的编码：路径与祖先集合

每个类别对应一条**从根到叶的路径**。祖先集合 `Aug(c)` = 路径上的所有节点（**不含根**）。

注意：`prohibitory` 这类**只到粗类**的标签也是合法标签——
它就是模块 00 说的「可降级输出」，路径长度为 1。

In [ ]:
TAXONOMY = {
    # 叶子（L2 细类）：path = (粗类, 细类)
    'speed_limit':      ('prohibitory', 'speed_limit'),
    'no_entry':         ('prohibitory', 'no_entry'),
    'no_left_turn':     ('prohibitory', 'no_left_turn'),
    'stop_giveway':     ('prohibitory', 'stop_giveway'),
    'ped_crossing':     ('warning',     'ped_crossing'),
    'sharp_curve':      ('warning',     'sharp_curve'),
    'road_work':        ('warning',     'road_work'),
    'go_straight':      ('mandatory',   'go_straight'),
    'min_speed':        ('mandatory',   'min_speed'),
    'direction_board':  ('guide',       'direction_board'),
    'vehicle_type_aux': ('auxiliary',   'vehicle_type_aux'),
    # **降级标签**：只到粗类（远处只看得出形状与颜色时使用）
    'prohibitory':      ('prohibitory',),
    'warning':          ('warning',),
    'mandatory':        ('mandatory',),
    'guide':            ('guide',),
}

def aug(c):
    '''祖先集合（含自身，不含根）。'''
    return set(TAXONOMY[c])

def depth(c):
    return len(TAXONOMY[c])

print(f"{'类别':<18s} {'路径':<34s} {'深度':>4s}")
for c in ['speed_limit', 'no_entry', 'ped_crossing', 'prohibitory']:
    print(f'{c:<18s} {" / ".join(TAXONOMY[c]):<34s} {depth(c):>4d}')

assert aug('speed_limit') == {'prohibitory', 'speed_limit'}
assert aug('prohibitory') == {'prohibitory'}
assert aug('speed_limit') & aug('no_entry') == {'prohibitory'}      # 同族：共享粗类
assert aug('speed_limit') & aug('ped_crossing') == set()            # 跨族：毫无重合
print('\n✅ 「同族错分」与「跨族错分」在祖先集合上有清晰的数学差别：')
print('   前者交集非空（至少粗类判对了），后者交集为空（形状和颜色都判错了）。')
print('   下一节把这个差别变成一个可以进 CI 的指标。')

## 5 · 层次感知评测：为什么「诚实的降级」必须被奖励

$$hP=\frac{|Aug(\hat y)\cap Aug(y)|}{|Aug(\hat y)|},\quad
  hR=\frac{|Aug(\hat y)\cap Aug(y)|}{|Aug(y)|},\quad
  hF=\frac{2\,hP\,hR}{hP+hR}$$

扁平准确率把所有错误都记 0 分。**hF 会区分三种错误，而且给「只输出粗类」比
「自信地答错细类」更高的分——这正是我们想要的激励。**

In [ ]:
def h_prf(pred, true):
    '''层次 precision / recall / F1（Kiritchenko et al., 2006）。'''
    ap, at = aug(pred), aug(true)
    inter = len(ap & at)
    hp = inter / len(ap)
    hr = inter / len(at)
    hf = 0.0 if (hp + hr) == 0 else 2 * hp * hr / (hp + hr)
    return hp, hr, hf

CASES = [
    ('speed_limit',  'speed_limit',  '完全正确'),
    ('prohibitory',  'speed_limit',  '**诚实的降级**：只输出粗类'),
    ('no_entry',     'speed_limit',  '同族错分：至少知道是禁令牌'),
    ('ped_crossing', 'speed_limit',  '跨族错分：形状与颜色都判错'),
]
print(f"{'预测':<16s} {'真值':<14s} {'扁平':>5s} {'hP':>6s} {'hR':>6s} {'hF':>6s}  解读")
for pred, true, note in CASES:
    hp, hr, hf = h_prf(pred, true)
    print(f'{pred:<16s} {true:<14s} {int(pred==true):>5d} '
          f'{hp:>6.2f} {hr:>6.2f} {hf:>6.2f}  {note}')

assert h_prf('speed_limit',  'speed_limit')[2] == 1.0
assert abs(h_prf('prohibitory', 'speed_limit')[2] - 2/3) < 1e-12
assert h_prf('no_entry',     'speed_limit')[2] == 0.5
assert h_prf('ped_crossing', 'speed_limit')[2] == 0.0
# **核心不等式**
assert (h_prf('prohibitory', 'speed_limit')[2]
        > h_prf('no_entry', 'speed_limit')[2]
        > h_prf('ped_crossing', 'speed_limit')[2])
print('\n✅ **核心不等式：0.67（诚实降级） > 0.50（同族错分） > 0.00（跨族错分）**')
print('   在扁平准确率下这三种情况全是 0 分 —— 于是模型没有任何理由输出「我只知道')
print('   这是禁令牌」，还不如赌一个细类。**指标的形状决定了模型的行为。**')

preds = ['speed_limit']*60 + ['prohibitory']*15 + ['no_entry']*15 + ['ped_crossing']*10
trues = ['speed_limit']*100
flat_acc = float(np.mean([p == t for p, t in zip(preds, trues)]))
mean_hf  = float(np.mean([h_prf(p, t)[2] for p, t in zip(preds, trues)]))
print(f'\n一批 100 条预测：扁平准确率 {flat_acc:.2f} · 平均 hF {mean_hf:.3f}')
assert abs(flat_acc - 0.60) < 1e-12
assert abs(mean_hf - (60*1.0 + 15*(2/3) + 15*0.5 + 10*0.0)/100) < 1e-12
print(f'   两个数差了 {mean_hf-flat_acc:.3f} —— 而这部分全部来自「模型其实部分答对了」的样本。')

## 6 · 标注一致性：先量化标签噪声，再谈模型精度

模拟两名认真的标注员独立标同一批图：
框有 ±1.5 px 的手抖、8% 的漏标、12% 的同族类别分歧。
**然后按尺寸分桶看一致性——这一步是关键，因为总体数字会掩盖小目标的灾难。**

In [ ]:
def iou_xyxy(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / union if union > 0 else 0.0

def iou_shift(side, delta):
    '''边长 side 的方框在 x,y 各偏移 delta 后与原框的 IoU（解析解）。'''
    inter = max(0.0, side - delta) ** 2
    return inter / (2 * side**2 - inter)

print('先看几何：同样 2 px 的分歧，不同尺寸的框会被判成什么')
for s_ in [8, 16, 32, 64]:
    v = iou_shift(s_, 2)
    tail = '❌ 按 IoU>=0.5 会被判成「不是同一个目标」' if v < 0.5 else ''
    print(f'  {s_:>3d}x{s_:<3d}  IoU={v:.3f}  {tail}')
assert abs(iou_shift(8, 2) - 36/92) < 1e-12
assert abs(iou_shift(64, 2) - 0.8842) < 1e-3
print('  → 标注抖动的**绝对**量级恒定（±1-2px），**相对**量级随尺寸反比放大。\n')

# —— 模拟两名标注员独立标同一批图
rng2 = np.random.default_rng(768)
N = 400
SIZES_POOL = [8, 12, 16, 24, 32, 48, 64]
sizes = rng2.choice(SIZES_POOL, size=N, p=[.22, .20, .18, .15, .11, .08, .06])
cx = rng2.uniform(100, 1800, N); cy = rng2.uniform(200, 700, N)
CLS = ['speed_limit', 'no_entry', 'ped_crossing', 'sharp_curve',
       'go_straight', 'direction_board']
clsA = rng2.choice(len(CLS), size=N)
A = [(cx[i]-sizes[i]/2, cy[i]-sizes[i]/2, cx[i]+sizes[i]/2, cy[i]+sizes[i]/2)
     for i in range(N)]

keep = rng2.random(N) > 0.08                     # 标注员 B 漏标 8%
jit  = rng2.normal(0, 1.5, size=(N, 4))          # 四个角各自 ±1.5px 手抖
B, clsB = [], []
for i in range(N):
    if not keep[i]:
        continue
    B.append(tuple(np.array(A[i]) + jit[i]))
    c = clsA[i]
    if rng2.random() < 0.12:                     # 12% 的同族类别分歧
        c = (c + 1) % len(CLS)
    clsB.append(c)

# 贪心匹配（IoU 从高到低，一对一）
cand = []
for i in range(len(A)):
    for j in range(len(B)):
        v = iou_xyxy(A[i], B[j])
        if v >= 0.5:
            cand.append((v, i, j))
cand.sort(reverse=True)
used_a, used_b, matched = set(), set(), []
for v, i, j in cand:
    if i in used_a or j in used_b:
        continue
    used_a.add(i); used_b.add(j); matched.append((i, j, v))

def bucket(s):
    return '<16px' if s < 16 else ('16-32px' if s < 32 else '>=32px')

tot_b = collections.Counter(bucket(s) for s in sizes)
mat_b = collections.defaultdict(list)
for i, j, v in matched:
    mat_b[bucket(sizes[i])].append(v)

print(f'标注员 A: {len(A)} 框 · 标注员 B: {len(B)} 框 · 匹配上 {len(matched)} 对\n')
print(f"{'尺寸桶':<10s} {'A 的框数':>9s} {'匹配率':>8s} {'匹配框平均 IoU':>16s}")
for k in ['<16px', '16-32px', '>=32px']:
    print(f'{k:<10s} {tot_b[k]:>9d} {len(mat_b[k])/tot_b[k]:>7.1%} '
          f'{np.mean(mat_b[k]):>16.3f}')

assert np.mean(mat_b['<16px']) < np.mean(mat_b['>=32px'])
assert np.mean(mat_b['<16px']) < 0.75, '小目标的框一致性天然很低'
assert len(mat_b['<16px'])/tot_b['<16px'] < len(mat_b['>=32px'])/tot_b['>=32px']

# 类别一致性：原始一致率 vs Cohen's kappa
ya = np.array([clsA[i] for i, j, v in matched])
yb = np.array([clsB[j] for i, j, v in matched])
po = float(np.mean(ya == yb))
pa = np.bincount(ya, minlength=len(CLS)) / len(ya)
pb = np.bincount(yb, minlength=len(CLS)) / len(yb)
pe = float((pa * pb).sum())
kappa = (po - pe) / (1 - pe)
print(f'\n类别一致率 p_o = {po:.3f} · 随机一致 p_e = {pe:.3f} · **Cohen κ = {kappa:.3f}**')
assert 0.6 < kappa < po, 'κ 扣掉了「碰巧一致」的部分，所以必然小于原始一致率'

small_iou = float(np.mean(mat_b['<16px']))
print(f'\n⚠️  **总体数字会骗人**：>=32px 桶的匹配 IoU 有 '
      f'{np.mean(mat_b[">=32px"]):.2f}，但 <16px 桶只有 {small_iou:.2f}。')
print('    这意味着：在小尺寸桶上用 IoU=0.5/0.75 评测，你测的一大半是**标注抖动**。')
print('✅ 可执行的纪律：动模型之前，先做一次双标注抽检，拿到三个数——')
print('   ① 指标提升的**噪声地板**（低于它的提升不值得立项）')
print('   ② 规范漏洞清单（分歧集中在哪几条规则上，那几条就是写得不够清楚的）')
print('   ③ 分尺寸桶的评测阈值依据（小桶必须放宽，或改用对尺度不敏感的度量）')

## ✏️ 练习 1：repeat factor sampling（LVIS 的做法）

实现 `repeat_factors(counts, t)`：

- 类别频率 $f_c = \mathrm{counts}_c / \sum \mathrm{counts}$
- 重复因子 $r_c = \max\left(1,\ \sqrt{t/f_c}\right)$（`counts_c == 0` 时记 `r_c = 1.0`）
- 重采样后的期望类别分布 $q_c \propto \mathrm{counts}_c \cdot r_c$（需归一化）

返回 `(r, q)` 两个 numpy 数组。

**注意那个平方根**：它让提升是温和的（尾类频率提升 ~$\sqrt{\cdot}$ 倍而不是被拉平），
这正是 repeat factor sampling 避免尾类过拟合的设计。

In [ ]:
def repeat_factors(counts, t=0.01):
    # TODO: ① 算频率 f  ② r = max(1, sqrt(t/f))，counts==0 时 r=1
    #       ③ q ∝ counts * r 并归一化   ④ 返回 (r, q)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
COUNTS_T = np.array([12000, 5000, 2000, 800, 300, 120, 60, 25, 10, 4])
f_T = COUNTS_T / COUNTS_T.sum()
r, q = repeat_factors(COUNTS_T, t=0.01)

assert np.all(r >= 1.0 - 1e-12), 'repeat factor 不能小于 1（不允许下采样 head）'
assert np.isclose(r[0], 1.0) and np.isclose(r[4], 1.0), 'f >= t 的类 r 恰好为 1'
assert abs(r[-1] - 7.1272) < 1e-3, f'最尾类 r 应约 7.127，得到 {r[-1]}'
assert np.all(np.diff(r) >= -1e-12), 'counts 递减 => r 必须递增（单调性）'
assert np.isclose(q.sum(), 1.0)
assert q[-1] > 6 * f_T[-1], '尾类占比应被显著提升'
assert q[0] < f_T[0], 'head 占比被相对压低'

print(f"{'类别':>4s} {'实例数':>8s} {'原频率':>9s} {'r':>7s} {'重采样后':>10s} {'提升':>7s}")
for i in range(len(COUNTS_T)):
    print(f'{i:>4d} {COUNTS_T[i]:>8,d} {f_T[i]:>8.4%} {r[i]:>7.3f} {q[i]:>9.4%} '
          f'{q[i]/f_T[i]:>6.2f}x')
print('\n✅ 练习 1 通过。三条结论：')
print('   ① f >= t 的类 r 恰为 1 —— **repeat factor 只抬尾巴，不压头部**')
print('   ② 提升是 sqrt 级的（尾类 7x 而不是 3000x）—— 把分布拉平会让尾类严重过拟合')
print('   ③ t 是唯一的旋钮：t 越大，被判为「尾」的类越多。常用取值 0.001~0.01')

## ✏️ 练习 2：尾类合并（层次结构最直接的用法）

实现 `merge_tail_classes(counts, parent, min_count)`：
把实例数 `< min_count` 的类别**并到它的父节点**，重复直到不能再合并
（父节点为 `None` 表示已到顶，即使不足 `min_count` 也保留）。

返回 `(new_counts, mapping)`：
- `new_counts`: `{最终标签: 实例数}`
- `mapping`: `{原类别: 最终标签}`（没被合并的类映射到自己）

**总实例数必须守恒**——这是这类操作的第一条不变量。

In [ ]:
def merge_tail_classes(counts, parent, min_count):
    # TODO: 循环：找到 count < min_count 且 parent 不为 None 的类，
    #       把它的 count 加到父节点上、从结果里删掉、更新 mapping；
    #       直到没有可合并的类为止。
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
C_IN = {'p/speed_60': 5000, 'p/speed_80': 3000, 'p/no_entry': 400,
        'p/no_uturn': 30, 'p/no_truck': 12,
        'w/ped': 800, 'w/slippery': 25, 'w/animals': 4,
        'm/turn_left': 600, 'm/min_speed': 8}
PARENT = {'p/speed_60': 'p', 'p/speed_80': 'p', 'p/no_entry': 'p',
          'p/no_uturn': 'p', 'p/no_truck': 'p',
          'w/ped': 'w', 'w/slippery': 'w', 'w/animals': 'w',
          'm/turn_left': 'm', 'm/min_speed': 'm',
          'p': None, 'w': None, 'm': None}

new_counts, mapping = merge_tail_classes(C_IN, PARENT, min_count=100)
EXPECT = {'p/speed_60': 5000, 'p/speed_80': 3000, 'p/no_entry': 400, 'p': 42,
          'w/ped': 800, 'w': 29, 'm/turn_left': 600, 'm': 8}
assert new_counts == EXPECT, new_counts
assert sum(new_counts.values()) == sum(C_IN.values()) == 9879, '总实例数必须守恒'
assert mapping['p/no_uturn'] == 'p' and mapping['w/animals'] == 'w'
assert mapping['p/speed_60'] == 'p/speed_60'
assert all(v >= 100 or PARENT.get(k) is None for k, v in new_counts.items())

print(f"{'最终标签':<14s} {'实例数':>8s}   来源")
for k in sorted(new_counts, key=lambda x: -new_counts[x]):
    src = sorted(o for o, t in mapping.items() if t == k)
    print(f'{k:<14s} {new_counts[k]:>8,d}   {", ".join(src)}')
print(f'\n类别数 {len(C_IN)} -> {len(new_counts)}，总实例数守恒 = {sum(new_counts.values()):,}')
print('\n✅ 练习 2 通过。两点必须理解：')
print('   ① 合并后的 `p`(42) 仍不足 100，但它已经没有父节点了 —— 真实系统里')
print('      这类桶应命名为 `other_prohibitory`，并**明确不对下游承诺细类**。')
print('   ② 合并**不是丢弃**：这些样本仍然监督着「这里有一块禁令牌」，')
print('      只是不再监督「是哪一种」。这正是层次标签最直接的价值。')

## ✏️ 练习 3：标注规范判定器

把讲解里的十二条规则中最核心的几条代码化。实现
`annotation_gate(ann, cfg)`，**按下列优先级顺序，先命中先返回**：

| 顺序 | 条件 | action | reason |
|---|---|---|---|
| 1 | `min(w,h) < cfg['floor_px']` | `skip` | `below_perception_floor` |
| 2 | `min(w,h) < cfg['min_px']` | `ignore` | `too_small` |
| 3 | `visible_ratio < cfg['skip_visible']` | `skip` | `mostly_occluded` |
| 4 | `visible_ratio < cfg['min_visible']` | `ignore` | `partially_occluded` |
| 5 | `key_region_visible` 为 False | `ignore` | `key_region_occluded` |
| 6 | `facing != 'front'` | `ignore` | `not_front_facing` |
| 7 | `readable` 为 False | `ignore` | `unreadable` |
| 8 | 以上都不命中 | `label` | `ok` |

返回 `{'action':…, 'reason':…, 'applies_to_ego': ann['applies_to_ego']}`。

⚠️ **`applies_to_ego` 绝不是过滤条件，只是透传的属性**——
「这块牌是否对自车生效」需要车道拓扑与规划路径，标注员和感知模型都没有这个上下文。

In [ ]:
GATE_CFG = {'floor_px': 4, 'min_px': 16, 'skip_visible': 0.3, 'min_visible': 0.6}

def annotation_gate(ann, cfg=GATE_CFG):
    # TODO: 按上表顺序判定，返回 {'action':…, 'reason':…, 'applies_to_ego':…}
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
def _ann(**kw):
    base = dict(w=40, h=40, visible_ratio=1.0, key_region_visible=True,
                facing='front', readable=True, applies_to_ego=True)
    base.update(kw); return base

CHECKS = [
    (_ann(),                              'label',  'ok'),
    (_ann(w=10, h=10),                    'ignore', 'too_small'),
    (_ann(w=3,  h=3),                     'skip',   'below_perception_floor'),
    (_ann(visible_ratio=0.45),            'ignore', 'partially_occluded'),
    (_ann(visible_ratio=0.20),            'skip',   'mostly_occluded'),
    (_ann(key_region_visible=False),      'ignore', 'key_region_occluded'),
    (_ann(facing='back'),                 'ignore', 'not_front_facing'),
    (_ann(facing='side'),                 'ignore', 'not_front_facing'),
    (_ann(readable=False),                'ignore', 'unreadable'),
    # **关键用例**：对向车道的牌，其它条件都正常 -> 照常标注！
    (_ann(applies_to_ego=False),          'label',  'ok'),
    # 优先级：又小又被挡，应先命中「小」
    (_ann(w=10, h=10, visible_ratio=0.2), 'ignore', 'too_small'),
]
print(f"{'用例':<46s} {'action':>8s}  reason")
for ann, exp_a, exp_r in CHECKS:
    got = annotation_gate(ann)
    assert got['action'] == exp_a and got['reason'] == exp_r, (ann, got, exp_a, exp_r)
    assert got['applies_to_ego'] == ann['applies_to_ego'], 'applies_to_ego 必须原样透传'
    desc = (f"{ann['w']}x{ann['h']}px vis={ann['visible_ratio']:.2f} "
            f"key={int(ann['key_region_visible'])} {ann['facing']} "
            f"read={int(ann['readable'])} ego={int(ann['applies_to_ego'])}")
    print(f'{desc:<46s} {got["action"]:>8s}  {got["reason"]}')

n_label = sum(annotation_gate(a)['action'] == 'label' for a, _, _ in CHECKS)
assert n_label == 2
print('\n✅ 练习 3 通过。三条最容易做错的规则：')
print('   ① **三档而不是两档**：<4px 不标、4-16px 标 ignore、>=16px 正常标。')
print('      只有「标/不标」两档时，小标志会变成**负样本**，模型被显式训练成')
print('      「看到 10px 的圆牌要判负」—— 远距离召回从此救不回来。')
print('   ② **关键区域可见性**优先于面积可见比例：限速牌边缘挡 80% 仍可读，')
print('      数字挡 20% 就完全不可读。面积比例只是代理指标。')
print('   ③ **对向车道的牌照常标注**。把「是否对自车生效」塞进标注/感知层，')
print('      等于永久丢掉做正确判断所需的证据。')

## ✏️ 练习 4：层次误分类代价矩阵

实现 `hierarchical_cost_matrix(classes, taxonomy)`：返回一个
`len(classes) x len(classes)` 的 numpy 矩阵，`C[i][j] = 1 - hF(classes[i], classes[j])`。

（可以直接复用上面的 `aug()`，或从 `taxonomy` 现取。）

做完你会发现一件漂亮的事：这个代价恰好等于
$\dfrac{d_i+d_j-2d_{\mathrm{LCA}}}{d_i+d_j}$，即**树上的归一化距离**——
`1 - hF` 与「最近公共祖先距离」是同一个东西的两种写法。

In [ ]:
def hierarchical_cost_matrix(classes, taxonomy):
    # TODO: 对每一对 (i, j) 算 1 - hF
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
CLASSES4 = ['speed_limit', 'no_entry', 'ped_crossing', 'prohibitory']
C = hierarchical_cost_matrix(CLASSES4, TAXONOMY)

assert C.shape == (4, 4)
assert np.allclose(np.diag(C), 0.0), '对角必须为 0（预测正确无代价）'
assert np.allclose(C, C.T), '代价矩阵必须对称'
assert abs(C[0, 1] - 0.50) < 1e-12, '同族错分（限速 vs 禁止驶入）代价 0.5'
assert abs(C[0, 2] - 1.00) < 1e-12, '跨族错分（限速 vs 注意行人）代价 1.0'
assert abs(C[0, 3] - 1/3)  < 1e-12, '降级到粗类代价仅 1/3'
assert C[0, 3] < C[0, 1] < C[0, 2], '降级 < 同族错分 < 跨族错分'

# 与「树上归一化 LCA 距离」等价
def tree_cost(a, b):
    da, db = len(TAXONOMY[a]), len(TAXONOMY[b])
    lca = len(aug(a) & aug(b))
    return (da + db - 2 * lca) / (da + db)
for i, a in enumerate(CLASSES4):
    for j, b in enumerate(CLASSES4):
        assert abs(C[i, j] - tree_cost(a, b)) < 1e-12, (a, b)

hdr = ''.join(f'{c[:12]:>14s}' for c in CLASSES4)
print(f'{"真值 / 预测":<18s}{hdr}')
for i, a in enumerate(CLASSES4):
    print(f'{a:<18s}' + ''.join(f'{C[i, j]:>14.3f}' for j in range(4)))
print('\n✅ 练习 4 通过。这个矩阵可以直接用在三个地方：')
print('   ① **代价敏感的损失**：把 CE 换成按 C 加权的损失，跨族错分惩罚更重')
print('   ② **代价敏感的评测**：报告「平均误分类代价」而不是「错误率」')
print('   ③ **badcase 优先级**：按 C[真值][预测] 排序，先看跨族错分')
print('   而且 1 - hF 恰好等于树上的归一化 LCA 距离 —— 两种写法，同一个东西。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def repeat_factors(counts, t=0.01):
    counts = np.asarray(counts, dtype=float)
    f = counts / counts.sum()
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.divide(t, f, out=np.full_like(f, np.inf), where=f > 0)
    r = np.maximum(1.0, np.where(counts > 0, np.sqrt(ratio), 1.0))
    q = counts * r
    return r, q / q.sum()

In [ ]:
# 练习 2 参考答案
def merge_tail_classes(counts, parent, min_count):
    cur = dict(counts)
    mapping = {c: c for c in counts}
    while True:
        victim = next((c for c in sorted(cur)
                       if cur[c] < min_count and parent.get(c) is not None), None)
        if victim is None:
            return cur, mapping
        tgt = parent[victim]
        cur[tgt] = cur.get(tgt, 0) + cur.pop(victim)
        for k, v in mapping.items():
            if v == victim:
                mapping[k] = tgt

In [ ]:
# 练习 3 参考答案
def annotation_gate(ann, cfg=GATE_CFG):
    short = min(ann['w'], ann['h'])
    if   short < cfg['floor_px']:                    a, why = 'skip',   'below_perception_floor'
    elif short < cfg['min_px']:                      a, why = 'ignore', 'too_small'
    elif ann['visible_ratio'] < cfg['skip_visible']: a, why = 'skip',   'mostly_occluded'
    elif ann['visible_ratio'] < cfg['min_visible']:  a, why = 'ignore', 'partially_occluded'
    elif not ann['key_region_visible']:              a, why = 'ignore', 'key_region_occluded'
    elif ann['facing'] != 'front':                   a, why = 'ignore', 'not_front_facing'
    elif not ann['readable']:                        a, why = 'ignore', 'unreadable'
    else:                                            a, why = 'label',  'ok'
    # ⚠️ applies_to_ego **只透传，不参与判定**
    return {'action': a, 'reason': why, 'applies_to_ego': ann['applies_to_ego']}

In [ ]:
# 练习 4 参考答案
def hierarchical_cost_matrix(classes, taxonomy):
    n = len(classes)
    C = np.zeros((n, n))
    for i, a in enumerate(classes):
        for j, b in enumerate(classes):
            ap, at = set(taxonomy[a]), set(taxonomy[b])
            inter = len(ap & at)
            hp, hr = inter / len(ap), inter / len(at)
            hf = 0.0 if (hp + hr) == 0 else 2 * hp * hr / (hp + hr)
            C[i, j] = 1.0 - hf
    return C

---
## 🧪 真实工程胶囊：类别体系 + 标注规范（可直接评审的一页纸）

下面这份 `RECIPE` 可以原样复制成项目里的 `taxonomy.yaml` / `annotation_spec.md` 骨架。
它把本模块的四件事——**层次类别树 / 属性表 / 十二条标注规则 / 划分与验收**——
落成可评审、可版本化的条目。

In [ ]:
RECIPE = r'''
# ============================================================
# TSR 类别体系与标注规范（骨架）   spec_version: 2024.1
# 每一条标注必须记录 spec_version；规范变更时必须评估是否需要回溯重标。
# ============================================================

region: CN                      # CN=GB 5768 / EU=Vienna / US=MUTCD —— **显式声明体系**
std_version: GB5768-2022        # 道路上长期并存新旧版图案：用属性区分，不新增类别

## 1. 层次类别树（L1 粗类 -> L2 细类）—— 类别表只放「决定用哪套能力」的信息
taxonomy:
  prohibitory:  [speed_limit, speed_cancel, no_entry, no_left_turn, no_uturn,
                 no_parking, no_overtaking, stop_giveway, yield_giveway, weight_limit]
  warning:      [ped_crossing, sharp_curve, road_work, slippery, school_zone,
                 falling_rocks, animals, narrow_road]
  mandatory:    [go_straight, turn_left, turn_right, keep_right, min_speed, roundabout]
  guide:        [direction_board, exit_board, distance_board, lane_board]
  auxiliary:    [vehicle_type_aux, time_range_aux, distance_aux, lane_aux]
  workzone:     [detour, lane_closed, flagger]
  vms:          [vms_speed, vms_text]        # LED 可变电子牌：视觉形态与实体牌完全不同

# **降级标签合法**：只给到 L1（如 prohibitory）是合法输出，评测用层次 hF 计分。

## 2. L3 属性（不进类别表！这里是长尾被消化掉的地方）
attributes:
  speed_value:    {type: int,  values: [5,10,15,20,30,40,50,60,70,80,90,100,110,120]}
  speed_unit:     {type: enum, values: [kmh, mph]}       # 跨区域必须显式
  variant:        {type: enum, values: [limit, cancel, minimum, section_start, section_end]}
  carrier:        {type: enum, values: [physical, vms]}
  vehicle_type:   {type: enum, values: [null, truck, bus, trailer, hazmat]}
  time_range:     {type: str,  example: "08:00-18:00"}
  distance_m:     {type: int,  example: 500}
  lane_id:        {type: int,  nullable: true}
  condition:      {type: enum, values: [normal, faded, damaged, graffiti, occluded]}
  facing:         {type: enum, values: [front, side, back]}   # 几何证据，供下游判定
  applies_to_ego: {type: bool, note: "标注员不判定；由下游用车道拓扑+规划路径判定"}
  group_id:       {type: int,  note: "组合牌：主牌与辅助牌共享同一个 group_id"}
  is_truncated:   {type: bool}
  spec_version:   {type: str}

## 3. 标注规则（十二条，按判定优先级）
gate:
  size:       {floor_px: 4, min_px: 16}      # <4 不标 / 4-16 ignore / >=16 正常标
  occlusion:  {skip_visible: 0.30, min_visible: 0.60,
               key_region_must_be_visible: true}   # 关键区域优先于面积可见比例
  truncation: clipped                        # 全项目统一 clipped（不用 amodal）+ is_truncated
  back_side:  {back: "标为 sign_back 独立类，作为难负样本", side: "ignore"}
  opposite_lane: "照常标注，只记录 facing / 横向位置；**不做是否生效的判定**"
  condition:  "褪色/破损/涂鸦照常标注 + condition 属性（用于分桶评测）"
  unreadable: "标框 + class=UNKNOWN + 忽略分类损失，**保留检测损失**"
  combo_sign: "主牌与辅助牌各标各的框 + 共享 group_id"
  vms:        "carrier=vms + 标注时刻读数 + is_variable=true"
  box_edge:   "含标志外框/白边，**不含**支撑杆件与背板（规范附图示）"
  not_a_sign: "广告牌图案 / 车身贴纸 / 前车导航屏 / 驾校教具 **不标为正样本**，
               但单独收集成难负样本集"
  duplicate:  "龙门架上重复出现的同内容标志：**都标**"

## 4. 划分与泄漏防护 —— 自建数据集最容易翻车的地方
split:
  unit: physical_sign_instance    # 绝不能按帧随机划分（同一块牌的相邻帧几乎一样）
  report_three_numbers: [by_frame, by_instance, by_city]
  # 三个数一起报，才说清模型泛化到了哪一层。只报最高的那个是自欺。

## 5. 标注验收 —— 先量标签噪声，再谈模型精度
annotation_qa:
  double_annotation_sample: 800          # 每批数据抽 800 张做双标
  thresholds:
    match_rate_ge_32px:  0.95
    match_rate_lt_16px:  0.70            # 小目标天然低，别定成 0.95
    mean_iou_ge_32px:    0.90
    mean_iou_lt_16px:    0.65            # ±2px 手抖在 8px 框上就是 IoU 0.39
    cohen_kappa_coarse:  0.90
    cohen_kappa_fine:    0.75            # 长尾场景必须用 kappa 而不是原始一致率
  noise_floor_note: "由双标注一致性推出的指标噪声地板；低于它的提升不立项。"

## 6. 公开数据集用途矩阵（按用途选，不是按大小选）
external_datasets:
  TT100K:   {use: [pretrain, structure_ablation], caveat: "腾讯街景车顶全景，无运动模糊，域差大"}
  MTSD:     {use: [cross_region_study],           caveat: "众包画质异质，相机内参未知"}
  BDD100K:  {use: [night_weather_detection],      caveat: "traffic sign 只有一个类，无细类"}
  GTSRB:    {use: [classifier_fast_ablation],     caveat: "**必须按 track 划分**，否则严重泄漏"}
  GTSDB:    {use: [smoke_test],                   caveat: "只有 900 张，极易过拟合"}
  DFG:      {use: [longtail_method_benchmark],    caveat: "200 类样本少，方法对照用"}
  CURE-TSD: {use: [degradation_bucket_eval],      caveat: "人工退化，非真实分布"}
'''
print(RECIPE)
for k in ['spec_version', 'region: CN', 'taxonomy', 'attributes', 'applies_to_ego',
          'floor_px', 'key_region_must_be_visible', 'group_id', 'not_a_sign',
          'physical_sign_instance', 'cohen_kappa_fine', 'external_datasets']:
    assert k in RECIPE, k
print('✅ 规范覆盖：体系声明 / 层次类别树 / 属性表 / 十二条规则 / '
      '划分防泄漏 / 标注 QA 阈值 / 数据集用途矩阵')

### 小结

- **类别体系比模型选型更早决定天花板**，而且它是不可逆资产：改一次细类定义，
  历史标注要么重标要么废弃。**「先随便定几个类，跑通再说」在 TSR 里是最贵的技术债。**
- **长尾要分成两半看**：*真实长尾*（「注意牲畜」本来就罕见）只能靠数据闭环解决；
  *人造长尾*（类别的笛卡尔积造出来的）可以靠层次设计**直接消除**。
  本课的账：**4,128 个扁平类 → 92 维层次输出，压缩 45×；
  新增一个限速值从「+160 类」变成「+1 个分箱」。**
- **幂律的残酷之处**：靠随机采集把最尾部的类补到 100 个实例，需要 **15 倍**于当前的数据量。
  **长尾只能靠定向挖掘（C58）+ 类别体系设计，不能靠「多采一点」。**
- **形状-颜色-语义映射在 GB 5768 / Vienna / MUTCD 里是三个不同的函数**：
  警告在中欧是三角形、在美国是黄色菱形；蓝色在中欧是强制指示、在美国是服务设施；
  蓝底指路牌在中国是普通道路、在德国是高速公路。
  **跨区域不是域适应问题，是任务重定义问题。**
- **层次评测的核心不等式：0.67（诚实降级）> 0.50（同族错分）> 0.00（跨族错分）。**
  扁平准确率把三者都记 0 分，于是模型没理由输出粗类。**指标的形状决定模型的行为。**
  而 `1 - hF` 恰好等于树上的归一化 LCA 距离。
- **标注要三档而不是两档**（不标 / ignore / 正常标）。只有两档时小标志会变成负样本，
  模型被训练成「看到 10 px 的圆牌判负」，远距离召回从此救不回来。
- **对向车道的牌照常标注，只输出几何证据。**「是否对自车生效」需要车道拓扑与规划路径；
  把这个判断塞进标注/感知层，等于永久丢掉做正确判断所需的证据。
- **动模型之前先量标签噪声**：小尺寸桶的匹配 IoU 只有 0.66、匹配率只有 74%。
  在这个桶上用 IoU=0.5/0.75 评测，测的一大半是标注抖动。
  **「你怎么知道 +0.5 mAP 不是噪声」的最佳答案是「我先量了噪声地板」。**
- **划分单位要与泛化目标一致**，并同时报告 by_frame / by_instance / by_city 三个数。

下一站：**模块 02 · TSR 系统设计：两级 vs 端到端** —— 级联召回是个乘法。